In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "research":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

os.chdir(PROJECT_ROOT)
print(f"Project root: {Path.cwd()}")
print(f"Config exists: {(Path('config') / 'config.yaml').exists()}")
print(f"Params exists: {Path('params.yaml').exists()}")


In [ ]:
from pathlib import Path

Path.cwd()


In [ ]:
# Already moved to the project root in the first cell.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str


In [ ]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config


In [ ]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = None

    def _get_tokenizer(self):
        if self.tokenizer is None:
            self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)
        return self.tokenizer

    def convert_examples_to_features(self, example_batch):
        tokenizer = self._get_tokenizer()
        input_encodings = tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        try:
            target_encodings = tokenizer(
                text_target=example_batch['summary'],
                max_length=128,
                truncation=True
            )
        except TypeError:
            with tokenizer.as_target_tokenizer():
                target_encodings = tokenizer(
                    example_batch['summary'],
                    max_length=128,
                    truncation=True
                )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        output_path = os.path.join(self.config.root_dir, 'samsum_dataset')
        if os.path.exists(os.path.join(output_path, "dataset_dict.json")):
            logger.info("Transformed dataset already exists at %s", output_path)
            return

        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(output_path)


In [ ]:
from pathlib import Path

print(f"Working directory: {Path.cwd()}")
print(Path("config/config.yaml").exists())
print(Path("params.yaml").exists())


In [ ]:
from pathlib import Path

ROOT_DIR = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

CONFIG_FILE_PATH = ROOT_DIR / "config" / "config.yaml"
PARAMS_FILE_PATH = ROOT_DIR / "params.yaml"

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e